# Compare pyLOCO with the PETRA III MATLAB result
This validation example repeats the standard one-iteration measured-ORM fit and compares the complete fitted parameter vector with the preserved MATLAB LOCO reference.

## 1. Load the measured-data workflow
The comparison uses exactly the same lattice, BPM rejection, corrector steps, uncertainties, fit blocks, and iteration settings as the standard PETRA III example.

In [1]:
from example_matlab_comparison import (HERE, compare_with_matlab, make_comparison_plots, print_comparison_summary)
from pyLOCO.measured_machine.workflow import fit_modes, model_orm, prepare_measurement, print_summary, run_fit
config_path = HERE / 'pyloco_config.yaml'
data = prepare_measurement(config_path)
print('Measured ORM:', data['orm'].shape)

Measured ORM: (470, 413)


## 2. Calculate the initial model response
This provides the usual scientific validation against the measurement in addition to the code-to-code parameter comparison.

In [2]:
initial_orm = model_orm(data)

## 3. Run the one-iteration pyLOCO fit
The MATLAB reference has 1500 entries and corresponds to the standard fit-block order in `pyloco_config.yaml`.

In [3]:
fit = run_fit(data)
coupling, constrained = fit_modes(fit)
assert not coupling and not constrained, 'MATLAB comparison requires the standard unconstrained YAML'
print('pyLOCO parameter-vector length:', len(fit['fit_results'][-1]))


==== Iteration 1/1 – LM ====
[Jacobian] Computing normal-quadrupole Jacobian (iteration 1)...
[calculate_quads_jacobian] Logs saved to '/private/var/folders/vj/crrgrwws3s902yfns0l06_rw0000gp/T/pyloco_petra_orm_b20e6jyj/logs/quad_jacobian_logs2.txt'
Normal quad Jacobian: 65.3 s
[Jacobian] Saved normal-quadrupole Jacobian to /var/folders/vj/crrgrwws3s902yfns0l06_rw0000gp/T/pyloco_petra_orm_b20e6jyj/jacobians/quads/J_quads_iter1_9.868774242500396e-05urad_-3000.0Hz.h5
   No outliers in the data set.
Initial Chi²: 2.4559e+04
  LM inner 1: chi² 6.5528e+02 (previous 2.4559e+04), λ=0.001
Chi² after correction: 6.5528e+02
LOCO LM completed! :).
pyLOCO parameter-vector length: 1500


## 4. Compare with MATLAB
The comparison first requires identical vector dimensions. It reports global RMS and maximum differences, then separates the vector into BPM gains, corrector calibrations, energy shifts, and quadrupole strengths so disagreement in one physical parameter family cannot be hidden by the full vector.

In [4]:
comparison = compare_with_matlab(data, fit)
print_comparison_summary(comparison)


pyLOCO and MATLAB parameter comparison
---------------------------------------
Parameters compared       : 1500
RMS absolute difference   : 6.891312064e-13
Maximum absolute difference: 3.841157508e-12
RMS relative difference   : 5.002478529e-08

Difference by parameter family
Parameter family                                  RMS        Maximum
Horizontal BPM gain                      6.429900e-13   3.518186e-12
Vertical BPM gain                        3.843424e-13   1.245004e-12
Horizontal corrector calibration         5.314096e-17   1.786901e-16
Vertical corrector calibration           5.185324e-17   2.627428e-16
Horizontal corrector energy shift        1.621939e-12   3.841158e-12
Quadrupole strength                      1.046757e-13   5.887235e-13


## 5. Report measurement fit quality and save comparison figures
Four figures are produced: the complete ordered vectors and difference, an agreement scatter, six physical parameter-family panels, and a logarithmic RMS/maximum difference summary by family. Agreement with MATLAB checks implementation consistency; the ORM residual still determines whether either fitted model explains the measured machine response.

In [5]:
output = make_comparison_plots(data, comparison)
print_summary(data, initial_orm, fit, coupling=coupling, constrained=constrained)
print('Figures:', output)


PETRA III measured-ORM fit
----------------------------------------------
Measured ORM shape : (470, 413)
Retained BPMs      : 235 per plane
Correctors         : 219 H, 194 V
Fitted parameters  : quads, hbpm_gain, vbpm_gain, hcor_cal, vcor_cal, HCMEnergyShift
ORM RMS before     : 154.339739 µm
ORM RMS after      : 49.797795 µm
Improvement        : 3.099x
Figures: /Users/musa/Desktop/pyLOCO/Examples/PETRAIII/output/matlab_comparison
